# Data Cleaning and Filtering

In [100]:
import pandas as pd

## Import Data Sets

In [ ]:
listenerDf = pd.read_csv('Global_Music_Streaming_Listener_Preferences.csv')
tracksDf = pd.read_csv('tracksDataset.csv')
spotifyDf = pd.read_csv('Most Streamed Spotify Songs 2024.csv', encoding='latin1')
artistsDf = pd.read_csv('artists.csv', dtype={'artist_lastfm': str, 'country_lastfm': str, 'tags_lastfm': str})

## Function to Fix the Data

In [ ]:
def cleaning(df, dfName):

    df.drop_duplicates(inplace=True)
    
    # replace any blanks with 0s
    df.fillna(0, inplace=True)

    # deleting unecessary columns
    if dfName == "listenerDf":
        df = df.drop(columns=['Repeat Song Rate (%)', 'Number of Songs Liked', 'Discover Weekly Engagement (%)'], errors = 'ignore')
        df=df.sort_values(by="Age")
        
    elif dfName == "tracksDf":
        df = df.drop(columns=['track_id', 'album_name', 'mode', 'time_signature', 'speechiness', 'liveness', 'instrumentalness'])
        df.columns.values[0] = 'Index'
        df.columns = [str(col).strip().capitalize() for col in df.columns]
        df.rename(columns={'Track_name': 'Track'}, inplace=True)
        df.rename(columns={'Artists': 'Artist'}, inplace=True)

        #convert miliseconds to minutes
        df["Duration_ms"] = df["Duration_ms"].apply(lambda ms: f"{int(ms)//60000:02d}:{(int(ms)//1000)%60:02d}")
        df.rename(columns={'Duration_ms': 'Duration'}, inplace=True)
        
    elif dfName == "artistsDf":
        #note: favoring Musicbrainz data due to better metadata
        df = df.drop(columns=['artist_lastfm', 'country_lastfm', 'tags_lastfm', 'listeners_lastfm', 'ambiguous_artist'])
        df.rename(columns={df.columns[0]: "Index"}, inplace=True)
        df["Index"] = range(1, len(df) + 1)

        df.rename(columns={'artist_mb': 'Artist'}, inplace=True)
        df.rename(columns={'country_mb': 'Artist Country'}, inplace=True)
        # drop duplicate artists, keep the first
        df = df.drop_duplicates(subset="Artist", keep="first")
        
    elif dfName == "spotifyDf":
        df = df.drop_duplicates(subset='ISRC')
        #note: only analyzing top 3 music platforms (Spotify, Apple, Amazon)
        df = df.drop(columns=['Album Name', 'Release Date', 'ISRC', 'Track Score' ])
        df = df.drop(columns=['AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Spotify Playlist Count'])
        df = df.drop(columns=['Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'TIDAL Popularity'])

    return df

## Call the Function

In [ ]:
listenerDf = cleaning(listenerDf, "listenerDf")
tracksDf = cleaning(tracksDf, "tracksDf")
spotifyDf = cleaning(spotifyDf, "spotifyDf")
artistsDf = cleaning(artistsDf, "artistsDf")